## Data Cleaning Summary

The Customer Segmentation project used three datasets from the Brazilian E-Commerce (Olist) dataset: Customers, Orders, and Order Payments.

The datasets were cleaned and merged in Power Query before the RFM analysis was performed.

Cleaning steps performed:

* Retained only relevant columns required for RFM analysis.
* Filtered the Orders dataset to include only delivered orders.
* Removed unnecessary delivery-related columns.
* Removed unnecessary payment-related columns.
* Verified that payment_value contained no missing values.
* Merged Customers and Orders using customer_id.
* Merged the resulting table with Payments using order_id.
* Retained customer_state for potential geographic analysis in Power BI.

The final Customer_RFM_Table contained 100,757 rows and 7 columns.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('cleaned_customer_rfm.csv')

In [ ]:
df.head()

,customer_id,customer_unique_id,customer_state,order_id,order_status,order_purchase_timestamp,payment_value
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,5/16/2017 15:05,146.87
1,0a8556ac6be836b46b3e89920d59291c,708ab75d2a007f0564aedd11139c7708,MG,b81ef226f3fe1789b1e8b2acac839d17,delivered,4/25/2018 22:01,99.33
2,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,SP,e481f51cbdc54678b7cc49136f2d6af7,delivered,10/2/2017 10:56,18.12
3,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,SP,e481f51cbdc54678b7cc49136f2d6af7,delivered,10/2/2017 10:56,2.00
4,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,SP,e481f51cbdc54678b7cc49136f2d6af7,delivered,10/2/2017 10:56,18.59


In [ ]:
df.shape

(100757, 7)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100757 entries, 0 to 100756
Data columns (total 7 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_id               100757 non-null  object 
 1   customer_unique_id        100757 non-null  object 
 2   customer_state            100757 non-null  object 
 3   order_id                  100757 non-null  object 
 4   order_status              100757 non-null  object 
 5   order_purchase_timestamp  100757 non-null  object 
 6   payment_value             100756 non-null  float64
dtypes: float64(1), object(6)
memory usage: 5.4+ MB


In [ ]:
df.isnull().sum()

,0
customer_id,0
customer_unique_id,0
customer_state,0
order_id,0
order_status,0
order_purchase_timestamp,0
payment_value,1


The cleaned customer dataset was imported into Google Colab for RFM analysis. A final quality check was performed to verify the dataset structure, data types, and missing values before calculating the RFM metrics.

The dataset contains 100,757 transaction records and 7 variables. One missing value was identified in the payment_value column and will be removed. The order_purchase_timestamp column will be converted to a datetime format to enable recency calculations.


In [ ]:
df = df.dropna(subset=['payment_value'])

df.isnull().sum()

,0
customer_id,0
customer_unique_id,0
customer_state,0
order_id,0
order_status,0
order_purchase_timestamp,0
payment_value,0


In [ ]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

df['order_purchase_timestamp'].dtype

dtype('<M8[ns]')

In [ ]:
snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

snapshot_date

Timestamp('2018-08-30 15:00:00')

In [ ]:
rfm = df.groupby('customer_unique_id').agg({

    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,

    'order_id': 'nunique',

    'payment_value': 'sum'

}).reset_index()

In [ ]:
rfm.columns = [

    'customer_unique_id',

    'Recency',

    'Frequency',

    'Monetary'

]

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
2,0000f46a3911fa3c0805444483337064,537,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [ ]:
rfm.shape

(93357, 4)

## Task 4: Customer Segmentation

After calculating the Recency, Frequency, and Monetary (RFM) metrics, customers were grouped into meaningful segments based on their purchasing behaviour.

The segmentation process helps identify customers who purchase frequently, spend the most, have recently interacted with the business, or may be at risk of becoming inactive.

These segments enable businesses to create targeted marketing strategies rather than applying the same approach to every customer.

The following customer groups will be created:

* Loyal Customers: Customers who purchase frequently and have made recent purchases.
* New Customers: Customers who have recently made their first purchase.
* Regular Customers: Customers with moderate purchasing activity.
* At Risk Customers: Customers whose purchasing activity has started to decline.
* Churn Risk Customers: Customers who have not made a purchase for a long period.


In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
)

rfm['M_score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1,2,3,4,5]
)

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,4
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4


In [ ]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(int)
    + rfm['F_score'].astype(int)
    + rfm['M_score'].astype(int)
)

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,4,9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1,6
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,4
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,4
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,7


In [ ]:
def customer_segment(score):

    if score >= 13:
        return 'Loyal Customer'

    elif score >= 10:
        return 'Regular Customer'

    elif score >= 8:
        return 'New Customer'

    elif score >= 6:
        return 'At Risk Customer'

    else:
        return 'Churn Risk Customer'

rfm['Customer_Segment'] = rfm['RFM_Score'].apply(customer_segment)

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Customer_Segment
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,4,9,New Customer
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1,6,At Risk Customer
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,4,Churn Risk Customer
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,4,Churn Risk Customer
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,7,At Risk Customer


In [ ]:
segment_distribution = (
    rfm['Customer_Segment']
    .value_counts()
)

segment_distribution

,count
Customer_Segment,
Regular Customer,31506
New Customer,27305
At Risk Customer,18663
Loyal Customer,8029
Churn Risk Customer,7854


The RFM scores were used to group customers into five distinct segments based on their purchasing behaviour.

The majority of customers were classified as Regular Customers and New Customers, indicating that many customers are either in the early stages of their relationship with the business or purchase occasionally.

A smaller group was identified as Loyal Customers, representing high-value customers who contribute significantly to the business and should be prioritized for retention strategies.

At Risk and Churn Risk Customers were also identified, highlighting opportunities for re-engagement campaigns and personalized marketing efforts to reduce customer attrition.


**Task 6: Behaviour** **Analysis**

In [ ]:
segment_analysis = rfm.groupby('Customer_Segment').agg({

    'Recency':'mean',

    'Frequency':'mean',

    'Monetary':'mean'

}).round(2)

segment_analysis

,Recency,Frequency,Monetary
Customer_Segment,,,
At Risk Customer,319.98,1.00,91.98
Churn Risk Customer,403.50,1.00,56.30
Loyal Customer,93.25,1.20,333.93
New Customer,248.90,1.01,139.68
Regular Customer,175.44,1.04,214.84


## Behaviour Analysis Findings

The behaviour analysis revealed clear differences among customer segments.

Loyal Customers had the lowest average recency (93.25 days), the highest purchase frequency (1.20 orders), and the highest average spending (333.93). This indicates that they are the most valuable customers and should be prioritized for retention strategies.

Regular Customers demonstrated moderate purchasing behaviour, with an average recency of 175.44 days and average spending of 214.84. These customers present an opportunity for upselling and personalized product recommendations.

New Customers had relatively recent purchases (248.90 days) but low purchase frequency, indicating that they are still in the early stages of their customer journey.

At Risk Customers showed declining engagement, with an average recency of 319.98 days and lower spending patterns. These customers may respond positively to targeted promotional campaigns.

Churn Risk Customers had the highest average recency (403.50 days) and the lowest average spending (56.30), indicating a high likelihood of customer attrition if no intervention is implemented.


In [ ]:
# Bring customer state back into the RFM table
state_df = df[['customer_unique_id', 'customer_state']].drop_duplicates()

rfm = rfm.merge(state_df, on='customer_unique_id', how='left')

In [ ]:
rfm.to_csv('customer_segments.csv', index=False)

print('Customer segmentation file created successfully.')

Customer segmentation file created successfully.


RFM Analysis Summary

The RFM analysis successfully transformed transactional data into customer-level insights by measuring customer recency, purchase frequency, and monetary value.

A total of 93,357 unique customers were segmented into five groups: Loyal Customers, Regular Customers, New Customers, At Risk Customers, and Churn Risk Customers.

This segmentation provides a foundation for targeted marketing strategies, customer retention initiatives, and personalized campaigns aimed at increasing customer lifetime value and reducing churn.
